# Local SLM LoRA Fine-Tuning for Expense Categorization (Apple Silicon MLX)

This notebook converts your cleaned Google Sheet CSV and verified `data/corrections.csv` into a ChatML dataset, fine-tunes `Qwen2.5-3B-Instruct-4bit` (or `1.5B` for 8GB/16GB Macs) using `mlx-lm` LoRA on Apple Metal, evaluates holdout accuracy, and fuses the LoRA adapter into `model/expense-slm-3b`.

In [2]:
# 1. Install MLX-LM and dependencies (run once)
%pip install -q -U mlx-lm pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 2. Load, Clean, and Format Dataset into Stratified ChatML JSONL
import json
import os
import re
import urllib.parse
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # Resolves to /Users/aloksingh/git/home-category-api
CSV_PATH = os.path.expanduser("~/Downloads/Expense Sheet - Categorized.csv")
CORRECTIONS_PATH = PROJECT_ROOT / "data" / "corrections.csv"
LORA_DATA_DIR = PROJECT_ROOT / "lora_data"
ADAPTER_DIR = PROJECT_ROOT / "adapters" / "expense-lora"
FUSED_MODEL_DIR = PROJECT_ROOT / "model" / "expense-slm-3b"

DIAGNOSTIC_PATTERN = (
    r'\b(?:blood\s*test|lab\s*test|diagnostic|x[\s-]*ray|mri|ct\s*scan|'
    r'ultrasound|sonography|ecg|echo|pathology|biopsy|cbc|thyroid|'
    r'lipid\s*profile|vitamin\s*[bd]|checkup|health\s*check|'
    r'thyrocare|lal\s*pathlabs|aarthi\s*scans|metropolis|orange\s*health|healthians)\b'
)

def clean_text(text: str) -> str:
    text = urllib.parse.unquote(str(text))
    text = re.sub(r'has been debited from account.*?to VPA', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'spent on your SBI Credit Card ending \d+ at', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'UPI transaction reference no.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'To check your available balance.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\d_]|[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# Load main sheet
df = pd.read_csv(CSV_PATH)
df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
df = df[~df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)]
df = df[pd.to_numeric(df['amount'], errors='coerce') > 1.0].copy()

# Merge human-verified corrections (upsampled 2x to reinforce edge cases)
if os.path.exists(CORRECTIONS_PATH):
    corr_df = pd.read_csv(CORRECTIONS_PATH)
    corr_df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
    corr_df = corr_df[
        ~corr_df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)
    ].copy()
    corr_df['amount'] = pd.to_numeric(corr_df['amount'], errors='coerce').fillna(0.0)
    df = pd.concat([df, corr_df, corr_df], ignore_index=True)

# Normalize taxonomy
df['Category'] = df['Category'].replace({'Shopping': 'Accessories', 'Househelp': 'House Help'})
combined_text = (df['Head'].fillna('') + ' ' + df['comment'].fillna('')).str.lower()
df.loc[combined_text.str.contains(DIAGNOSTIC_PATTERN, regex=True, na=False), 'Category'] = 'Medical'

counts = df['Category'].value_counts()
df = df[df['Category'].isin(counts[counts >= 5].index)].copy()

VALID_CATEGORIES = sorted(df['Category'].unique().tolist())
SYSTEM_PROMPT = (
    "Classify the Indian household expense into exactly one of these categories: "
    + ", ".join(VALID_CATEGORIES)
    + ". Reply with ONLY the category name."
)

def format_user_prompt(head: str, comment: str, amount: float) -> str:
    c_clean = clean_text(comment) if pd.notna(comment) and str(comment).strip() else "None"
    h_clean = str(head).strip()
    amt_str = f"₹{float(amount):.2f}" if float(amount) > 1.0 else "Unknown"
    return f"Head: {h_clean} | Comment: {c_clean} | Amount: {amt_str}"

def to_chatml_record(row) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_user_prompt(row['Head'], row['comment'], row['amount'])},
            {"role": "assistant", "content": str(row['Category']).strip()},
        ]
    }

train_df, valid_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Category'])

os.makedirs(LORA_DATA_DIR, exist_ok=True)
for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    out_path = os.path.join(LORA_DATA_DIR, f"{split_name}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in split_df.iterrows():
            f.write(json.dumps(to_chatml_record(row), ensure_ascii=False) + "\n")

print(f"Prepared {len(train_df)} train and {len(valid_df)} validation rows across {len(VALID_CATEGORIES)} categories.")
print("Sample prompt:", to_chatml_record(train_df.iloc[0])["messages"][1]["content"])

/var/folders/v0/gz626t394gj6jhy0b_p9pjyc0000gn/T/ipykernel_59056/1094397047.py:35: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
/var/folders/v0/gz626t394gj6jhy0b_p9pjyc0000gn/T/ipykernel_59056/1094397047.py:42: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  corr_df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)


Prepared 6369 train and 1125 validation rows across 23 categories.
Sample prompt: Head: Vegetable | Comment: None | Amount: ₹100.00


In [5]:
# 3. Run LoRA Fine-Tuning on Apple Silicon GPU
# Tip: Use 'mlx-community/Qwen2.5-1.5B-Instruct-4bit' on 8GB/16GB Macs, or 'Qwen2.5-3B-Instruct-4bit' on 24GB+ Macs
BASE_MODEL = "mlx-community/Qwen2.5-3B-Instruct-4bit"
ADAPTER_DIR = "adapters/expense-lora"

!python -m mlx_lm.lora \
  --model {BASE_MODEL} \
  --train \
  --data {LORA_DATA_DIR} \
  --iters 500 \
  --batch-size 4 \
  --learning-rate 1e-4 \
  --num-layers 8 \
  --steps-per-eval 100 \
  --adapter-path {ADAPTER_DIR}

Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files:   0%|                                  | 0/9 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B /   785B            

Fetching 9 files:  11%|██▉                       | 1/9 [00:00<00:02,  3.77it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0%|       |   785B / 1.67MB            
Reconstructing (incomplete total...):   0%|       |   785B / 1.67MB            
Reconstructing (incomplete total...):   0%|       | 1.39kB / 1.67MB            
Reconstructing (incomplete total...):   0%|       | 2.00kB / 1.74GB            
Reconstructing (incomplete total...):   0%|       | 1.67MB / 1.74GB        

In [6]:
# 4. Evaluate Holdout Accuracy & Check for Any Invalid Category Hallucinations
import json
from mlx_lm import load, generate
from sklearn.metrics import classification_report, accuracy_score

model, tokenizer = load(BASE_MODEL, adapter_path=ADAPTER_DIR)

valid_records = []
with open(os.path.join(LORA_DATA_DIR, "valid.jsonl"), "r", encoding="utf-8") as f:
    for line in f:
        valid_records.append(json.loads(line))

y_true, y_pred = [], []
hallucinated = []

for rec in valid_records:
    messages = rec["messages"][:2]  # system + user
    expected = rec["messages"][2]["content"]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=12, verbose=False).strip()

    y_true.append(expected)
    y_pred.append(pred)
    if pred not in VALID_CATEGORIES:
        hallucinated.append((messages[1]["content"], expected, pred))

print(f"Holdout Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print(f"Out-of-vocabulary predictions: {len(hallucinated)}")
print(classification_report(y_true, y_pred, zero_division=0))

/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2036.29it/s]


Holdout Accuracy: 92.71%
Out-of-vocabulary predictions: 0
               precision    recall  f1-score   support

  Accessories       0.99      0.91      0.95        76
   Appliances       0.29      0.67      0.40         3
    Baby Care       0.57      0.77      0.66        26
        Bills       0.97      0.96      0.96        67
  Celebration       1.00      0.45      0.62        11
    Education       0.92      0.92      0.92        26
  Electronics       1.00      1.00      1.00        10
Entertainment       1.00      1.00      1.00         2
      Fitness       0.70      1.00      0.82         7
 Food Outside       0.94      1.00      0.97       159
         Fuel       1.00      1.00      1.00        20
    Furniture       1.00      0.50      0.67         2
         Gift       0.69      0.93      0.79        29
      Grocery       0.98      0.95      0.97       368
     Grooming       0.97      0.91      0.94        34
   House Help       0.97      1.00      0.98        30
    Ho

In [7]:
# 5. Edge-Case Regression Sanity Check
edge_cases = [
    ("Dishwasher", "", 459.0, "Grocery"),
    ("Dishwasher", "INR 32472 at Flipkart In on 10-10-2024", 32472.0, "Appliances"),
    ("Hotter recho", "149.00 debited to VPA hotstaronline@ybl JioHotstar", 149.0, "Bills"),
    ("ऑटो", "", 107.0, "Travel"),
    ("Detergent powder", "", 666.0, "Grocery"),
    ("Lunch", "COMPASS INDIA FOOD SERVICES PR", 630.0, "Food Outside"),
    ("Blood test", "Thyrocare", 1200.0, "Medical"),
]

for head, comment, amount, expected in edge_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_user_prompt(head, comment, amount)},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=12, verbose=False).strip()
    status = "PASS" if pred == expected else "FAIL"
    print(f"[{status}] {head:<20} (₹{amount:>8.2f}) -> Expected: {expected:<14} | Actual: {pred}")

[FAIL] Dishwasher           (₹  459.00) -> Expected: Grocery        | Actual: Appliances
[PASS] Dishwasher           (₹32472.00) -> Expected: Appliances     | Actual: Appliances
[PASS] Hotter recho         (₹  149.00) -> Expected: Bills          | Actual: Bills
[PASS] ऑटो                  (₹  107.00) -> Expected: Travel         | Actual: Travel
[FAIL] Detergent powder     (₹  666.00) -> Expected: Grocery        | Actual: Baby Care
[PASS] Lunch                (₹  630.00) -> Expected: Food Outside   | Actual: Food Outside
[PASS] Blood test           (₹ 1200.00) -> Expected: Medical        | Actual: Medical


In [8]:
# 6. Fuse LoRA Adapter into a Standalone Model Directory
FUSED_MODEL_DIR = "model/expense-slm-3b"

!python -m mlx_lm.fuse \
  --model {BASE_MODEL} \
  --adapter-path {ADAPTER_DIR} \
  --save-path {FUSED_MODEL_DIR}

print(f"Fused standalone SLM saved to {FUSED_MODEL_DIR}")

Calling `python -m mlx_lm.fuse...` directly is deprecated. Use `mlx_lm.fuse...` or `python -m mlx_lm fuse ...` instead.
Loading pretrained model
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 1830.33it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
README.md: 100%|██████████████████████████████| 755/755 [00:00<00:00, 3.56MB/s]
Fused standalone SLM saved to model/expense-slm-3b


In [9]:
# 7. Dequantize, Fuse LoRA Adapter, Build Qwen2.5 ChatML Modelfile, and Import into Ollama
import shutil
import subprocess
import sys

FUSED_HF_DIR = PROJECT_ROOT / "training" / "slm" / "model" / "expense-slm-hf"
MODELFILE_PATH = PROJECT_ROOT / "training" / "slm" / "model" / "Modelfile"
OLLAMA_MODEL_NAME = "expense-categorizer"

# 1. Fuse LoRA adapter and dequantize weights into HF safetensors for Ollama conversion
fuse_cmd = [
    sys.executable, "-m", "mlx_lm", "fuse",
    "--model", BASE_MODEL,
    "--adapter-path", str(ADAPTER_DIR),
    "--save-path", str(FUSED_HF_DIR),
    "--dequantize",
]
subprocess.run(fuse_cmd, check=True)

# 2. Write Modelfile with Qwen2.5 ChatML TEMPLATE, stop tokens, and exact SYSTEM prompt
modelfile_content = f"""FROM ./{FUSED_HF_DIR.name}

TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
\"\"\"

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.0
PARAMETER num_predict 8

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")
print(f"Wrote ChatML Modelfile to {MODELFILE_PATH}")

# 3. Quantize directly to Q4_K_M (~1.9 GB) and register in Ollama
subprocess.run(
    ["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(MODELFILE_PATH), "-q", "Q4_K_M"],
    cwd=str(MODELFILE_PATH.parent),
    check=True,
)
print(f"Registered Ollama model: {OLLAMA_MODEL_NAME}")

# 4. Reclaim ~6 GB disk space once imported into Ollama
shutil.rmtree(FUSED_HF_DIR, ignore_errors=True)
print(f"Cleaned up temporary HF directory: {FUSED_HF_DIR}")

Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 19.03it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/mlx_lm/__main__.py", line 6, in <module>
    cli.main()
  File "/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/mlx_lm/cli.py", line 40, in main
    submodule.main()
  File "/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/mlx_lm/fuse.py", line 71, in main
    model, tokenizer, config = load(
                               ^^^^^
  File "/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/mlx_lm/utils.py", line 697, in load
    model = load_adapters(model, adapter_path)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/mlx_lm/utils.py", 

CalledProcessError: Command '['/Users/aloksingh/git/home-category-api/.venv-slm/bin/python', '-m', 'mlx_lm', 'fuse', '--model', 'mlx-community/Qwen2.5-3B-Instruct-4bit', '--adapter-path', '/Users/aloksingh/git/home-category-api/adapters/expense-lora', '--save-path', '/Users/aloksingh/git/home-category-api/training/slm/model/expense-slm-hf', '--dequantize']' returned non-zero exit status 1.

In [11]:
# 8. Smoke-Test Ollama Local Endpoint with Grammar-Constrained Output
import json
import urllib.request

payload = {
    "model": OLLAMA_MODEL_NAME,
    "stream": False,
    "format": {
        "type": "string",
        "enum": VALID_CATEGORIES,
    },
    "messages": [
        {
            "role": "user",
            "content": format_user_prompt(
                "Hotter recho",
                "149.00 debited to VPA hotstaronline@ybl JioHotstar",
                149.0,
            ),
        }
    ],
}

req = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

with urllib.request.urlopen(req, timeout=15) as resp:
    result = json.loads(resp.read().decode("utf-8"))
    print("Ollama Prediction:", result["message"]["content"])


Ollama Prediction: "Bills"
